In [1]:
import io

import requests
import zipfile
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import time

from pandas import Series
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
DATAPATH = PROJECT_ROOT / "src/database/BindingDB_All_202605_tsv.zip"
NOTEBOOK_DATAFOLDER = PROJECT_ROOT / "notebook_database"


In [2]:
# Goal: get protein sequence + smiles pairs
# task: get pocket patch proteins - need to isolate pocket sequence via UniProt + PDB
# BindingDB: Protein-Ligand affinity data + UniProt IDs
# PDB: will contain information on proteins close to drug (6-8 A)

In [3]:
# Data settings
target_fields = [
    "Ligand SMILES",
    "PDB ID(s) of Target Chain 1",
    "Ki (nM)",  # inhibitor binding strength -> both Ki and Kd are good since we want POCKETS
]

nm_threshold = 100   # isolate high affinity pairs
chunksize = 100000
pdb_resolution = 2.5

In [4]:
chunks = []

with zipfile.ZipFile(DATAPATH) as z_fil:
    tsv_data = [f for f in z_fil.namelist() if f.endswith(".tsv")][0]
    with z_fil.open(tsv_data) as f:
        reader = pd.read_csv(
            f,
            sep="\t",
            dtype=str,
            usecols=target_fields,
            low_memory=False,
            on_bad_lines="skip",
            chunksize=chunksize,
        )
        for chunk in reader:
            filtered = chunk[target_fields]
            chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)

In [ ]:
# I only want potent binders
# print(df.shape)  # 3176528, 4

# apply potency filter -> can adjust nm_threshold down the line
df["Ki (nM)"] = pd.to_numeric(df["Ki (nM)"], errors="coerce")
potent_df = df[df["Ki (nM)"] <= nm_threshold]
# print(potent_df.shape)   # 297264, 4

potent_df.head(3)

In [6]:
unique_smiles_strings = potent_df["Ligand SMILES"].dropna().unique()

all_smiles = set()
for smile in unique_smiles_strings:
    all_smiles.add(smile)

print(len(all_smiles))  # 155343
unique_smiles_strings = pd.DataFrame(all_smiles)
unique_smiles_strings.to_csv("notebook_database/unique_smiles_strings.csv")

155343


In [7]:
unique_pdb_strings = potent_df["PDB ID(s) of Target Chain 1"].dropna().unique()
# print(len(unique_pdb_strings))  # --> 1985 string clusters

all_pdb_ids = set()

for pdb_string in unique_pdb_strings:
    for pid in pdb_string.split(","):
        all_pdb_ids.add(pid.strip())

# print(len(all_pdb_ids)) # --> 30259

In [ ]:
# this took me 2 hours to run but im pretty sure with the multithreading thing it can be accelerated to like 5 mins. idk how right now though maybe someone can help figure that out

def get_pocket(pdb_id):
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id.upper()}"
    try:
        req = requests.get(url, timeout=10)
        if req.status_code != 200:
            return None
        data = req.json()

        req_data = data.get("rcsb_entry_info", {})

        return {
            "pdb_id": pdb_id.upper(),
            "resolution": req_data.get("resolution_combined", [999])[0],
            "ligand_count": req_data.get("nonpolymer_entity_count", 0),
        }

    except Exception as e:
        print(f"Couldn't get info on {pdb_id} : {e}")
        return None


def pdb_filters(all_pdb_ids, delay=0.1):
    result_df = []
    for pdb_id in tqdm(all_pdb_ids, total=len(all_pdb_ids), desc="Validating PDBs + Building Cache"):
        meta_data = get_pocket(pdb_id)

        # apply filters -> want relatively high resolution and a sequence that's acc bound to smth
        if meta_data is None:
            continue

        time.sleep(delay)

        if meta_data["resolution"] > 2.5:
            continue
        if meta_data["ligand_count"] == 0:
            continue

        result_df.append({
            "PDB ID": pdb_id,
        })

    return pd.DataFrame(result_df)

pdb_df = pdb_filters(all_pdb_ids=all_pdb_ids, delay=0.1)
pdb_df.to_csv("notebook_database/unique_pdb_cache.csv")

In [46]:
# Ok so pdb_df is gonna contain all valid unique pdb IDs
# potent_df is gonna contain all SMILES
# going to need to switch to mmCIF but that's fine - same 4 char codes
# isolate atoms within 8 angstroms of ligand - look for LIG comp_id
# we're getting relative position vectors to the centroid of the ligand
# this means in data generation, we would first have to compute the SMILES's 3d structure
#
# training: we're passing relative coords. to a centroid that's already calc., model would learn this implicitly during training via cross attention
# probably wouldn't hurt to pass centroid vector through regardless - we'll see
# save it just in case ^^
# so far the only reason this severely matters is data visualization, gonna need to relatively position the AA's around a calculated centroid
# would probably have an entirely different data visualizer module

# lemme just place this here to get the functions defined
# will call on it using thread in another cell

# notes from biopython cookbook
# structure consists of models
# model consists of chains
# chain consists of residues
# residue consists of atoms

# atom: atom obj: atom id simply is atom name
# Ca (alpha) atoms are called '.CA.'
from Bio.PDB.MMCIFParser import MMCIFParser
from Bio.PDB import Selection
from Bio.PDB.Polypeptide import PPBuilder

# load all unique pdb ids as a list
pdb_df = pd.read_csv("notebook_database/unique_pdb_cache.csv")
pdb_list = pdb_df["PDB ID"].to_list()

# Residues to exclude: waters, common ions, solvents
EXCLUDE_HETS = {
    "HOH", "WAT", "DOD",  # water
    "NA", "K", "MG", "CA", "ZN", "FE", "MN", "CU", "CL", "BR", "I",  # ions
    "SO4", "PO4", "GOL", "EDO", "PEG", "MPD", "DMS", "ACT", "ACE",   # solvents/buffers
    "NO3", "CO3", "EPE", "MES", "TRS", "HEZ"
}

AA_codetoletter = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
    "MSE": "selM",  # selenomethionine: gonna need to do some literature on these later. ok thank god they're not included in anything
    "SEP": "phoS",  # phosphoserine
    "TPO": "phoT",  # phosphothreonine
    "PTR": "phoY",  # phosphotyrosine
    "HYP": "hydP",  # hydroxyproline
}

ANGSTROM_THRESHOLD = 8.0


def fetch_struct(pdb_id: str):
    """I want to avoid disk writing, it's like 20k entries"""
    url = f"https://files.rcsb.org/download/{pdb_id.lower()}.cif"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return None

    cif_handle = io.StringIO(r.text)
    parser = MMCIFParser(QUIET=True)
    struct = parser.get_structure(pdb_id, cif_handle)
    return struct

def get_ligand_atoms(struct):
    """Extract HETATM belonging to drug-like ligands"""
    ligand_atoms = []
    for model in struct:
        residues = model.get_residues()
        for residue in residues:
            if residue.id[0].startswith("H_"):
                resname = residue.resname.strip()
                if resname not in EXCLUDE_HETS:
                    for atom in residue.get_atoms():
                        if atom.element != "H":
                            ligand_atoms.append((atom, residue))

    return ligand_atoms


def get_ligand_data(ligand_atoms):
    """get the geometric center (avg. pos) of all atoms within the ligand mol"""
    coords = np.array([atom.get_vector().get_array()
                       for atom, _ in ligand_atoms])
    return coords, coords.mean(axis=0)

def get_pocket_residues(struct, ligand_atoms, threshold: float = ANGSTROM_THRESHOLD):
    """
    for each protein residue, get their min distance to any ligand atom
    return residues within angstrom_threshold, get alpha carbon vector relative to ligand centroid
    :param struct:
    :param ligand_atoms:
    :param threshold:
    :return:
    """
    if not ligand_atoms:
        return []

    ligand_coords, ligand_centroid = get_ligand_data(ligand_atoms)

    AA_ids = []
    threeD_coords = []


    for model in struct:
        # residue.id is a tuple of (hetflag, sequence_number, insertion_code)
        for residue in model.get_residues():
            if residue.id[0] != " ":  # blank for standard amino and nucleic acids
                continue
            if "CA" not in residue:  # this is where we get the coordinate vector from
                continue

            # collect 3D coordinates of every non-hydrogen atom in the residue
            # cleaner, more efficient + doesn't contribute unique topological information
            r_records = np.array([
                atom.get_vector().get_array()
                for atom in residue.get_atoms()
                if atom.element != "H"
            ])
            # don't focus just on alpha Carbon at this stage since we want all proteins within that threshold distance FIRST

            # r_records is shape (N_res_atoms, 3)
            # ligand_coords is shape (N_lig_atoms, 3)
            # np.newaxis reshapes them so NumPy broadcasts subtraction across every pair
            # basically multiplying the matrices so they line up for subtraction
            diffs = r_records[:, np.newaxis, :] - ligand_coords[np.newaxis, :, :]
            min_dist = np.sqrt((diffs**2).sum(axis=2)).min()

            if min_dist <= threshold:
                ca_coord = residue["CA"].get_vector().get_array()
                relative_pos = ca_coord - ligand_centroid  # obtain AA vector relevant to ligand

                AA_ids.append(AA_codetoletter[str(residue.resname)])
                threeD_coords.append(relative_pos)

    # property dict
    pocket_patch_residues = {
        "AA_IDs": AA_ids,   # indices align
        "3d_struct": threeD_coords
    }

    return pocket_patch_residues


def pdb_miner(pdb_id: str):
    try:
        struct = fetch_struct(pdb_id)
        if struct is None:
            return None

        ligand_atoms = get_ligand_atoms(struct)
        if not ligand_atoms:
            return None

        residues = get_pocket_residues(struct, ligand_atoms)

        if not residues["AA_IDs"]:
            return None

        result = {
            "PDB ID": pdb_id,
            "pocket_AA_data": residues,
        }

        return result

    except Exception as e:
        print(f"Couldn't get data on {pdb_id}: {e}")
        return None

In [ ]:
# ok use this to run the above functions

from concurrent.futures import ThreadPoolExecutor, as_completed

def pocket_pipeline(pdb_ids: list, max_workers=16):
    AA3D_df = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(pdb_miner, pdb_id): pdb_id for pdb_id in pdb_ids}

        for future in tqdm(as_completed(futures), total=len(pdb_ids)):
            result = future.result()
            if result:
                AA3D_df.append(result)

    return pd.DataFrame(AA3D_df)

# bottleneck is network I/O, so batch processing won't do much. save that for training / tokenizing
# timelog: 100%|██████████| 20420/20420 [1:41:32<00:00,  3.35it/s]
AA3D_df = pocket_pipeline(pdb_ids=pdb_list, max_workers=16)

In [53]:
AA3D_df.to_csv("notebook_database/AA3D_df.csv")

In [ ]:
# MILESTONE 1: cleaned/processed + isolated protein binding pocket coordinates for all unique PBD IDs

In [5]:
AA3D_df = pd.read_csv("notebook_database/AA3D_df.csv")
# keys: PDB ID and pocket_AA_data
#                  |--> within pocket_AA_data there is:
#                   AA_IDs and 3d_struct -> amino acid code and 3d vectors

In [32]:
from rdkit import Chem
from map4 import MAP4
# look at study one molecular fingerprint to rule them all: drugs, biomolecules
# can always do a double take later

mol_dim = 1024
map4_fprinter = MAP4(
    dimensions=mol_dim,
    radius=2,
    include_duplicated_shingles=True,
)

def mol_from_smiles(smiles):
    """Extract molecular fingerprint from singular SMILES"""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        map4_fp = map4_fprinter.calculate(mol=mol)

        return map4_fp

    except Exception as e:
        return None


## TEST BLOCK
# test_smiles = "COc1nc(OC[C@H]2C[C@@H]2c2ccc3ncccc3n2)nc(NCc2cnn(C)c2)c1C |r|"
# test_fp = mol_from_smiles(test_smiles)

# test_fp  #array([1, 0, 1, ..., 0, 0, 0], shape=(1024,), dtype=uint8)

In [ ]:
unique_smiles_strings_df = pd.read_csv("notebook_database/unique_smiles_strings.csv")

MOLFP_df = []
idxs = 0
fails = 0
for _, row in tqdm(unique_smiles_strings_df.iterrows(), total=len(unique_smiles_strings), desc="Generating Fingerprints"):
    smiles = str(row["0"])
    fingerprint = mol_from_smiles(smiles=smiles)
    if fingerprint is None:
        fails += 1
        continue

    MOLFP_df.append({
        "smiles_str": str(smiles),
        "map4_fp": fingerprint,
    })
    idxs += 1

failrate = float(fails / idxs)  # check for systemic errors
print(failrate)  # 0.029% rounded up so we're good

In [36]:
MOLFP_df = pd.DataFrame(MOLFP_df)
MOLFP_df.to_csv("notebook_database/molfp_df.csv")

In [ ]:
# MILESTONE 2: obtained fingerprints for all unique SMILES

In [20]:
# one more thing to do: create training information csv
# col 1: all unique PDB IDs
# col 2: list of all SMILES tested on that protein
# mental note: remember they're all here because they had high binding affinity

initial_info = df.dropna(subset=["PDB ID(s) of Target Chain 1"])
unique_smiles = pd.read_csv("notebook_database/unique_smiles_strings.csv")
unique_pdb_ids = pd.read_csv("notebook_database/unique_pdb_cache.csv")

# initial_info.columns
# ['Ligand SMILES', 'PDB ID(s) of Target Chain 1', 'Ki (nM)']
# PDB column contains a string cluster of all PDB IDs for that - separated ','

# unique_smiles: smile strings are in unique_smiles["0"]...

# unique_pdb_ids: PDB IDs are in unique_pdb_ids["PDB ID"]


def get_all_pairs():
    """
    Iterate through the unique pdb ids first
    Obtain each ID's SMILES hit through 'initial_info'
    Doing this because we want to train test split by protein not molecule
    Wanna see if it'll generate proteins that max. biochemical / structural properties required to bind this molecule? or if it overtrained to see one specific pattern
    :return:
    """
    unique_pdb_set = set(unique_pdb_ids["PDB ID"].astype(str).str.strip())
    unique_smiles_set = set(unique_smiles["0"].astype(str).str.strip())

    df = initial_info.dropna(subset=["PDB ID(s) of Target Chain 1"]).copy()
    # Filter to only SMILES we actually want / have now
    df = df[df["Ligand SMILES"].isin(unique_smiles_set)]

    # Explode multi-PDB column
    df["pdb_list"] = df["PDB ID(s) of Target Chain 1"].str.split(",")
    exploded = df.explode("pdb_list").copy()

    exploded["pdb_list"] = exploded["pdb_list"].str.strip()

    # Keep only relevant PDBs and relevant SMILES (already filtered)
    exploded = exploded[exploded["pdb_list"].isin(unique_pdb_set)]
    exploded = exploded.drop_duplicates(subset=["pdb_list", "Ligand SMILES"])

    # Group SMILES per PDB
    pdb_to_smiles = exploded.groupby("pdb_list")["Ligand SMILES"].apply(list)

    # Create final dataframe with ALL unique PDBs
    result = pd.DataFrame({
        "PDB_ID": list(unique_pdb_set)
    })

    result["SMILES_hits"] = result["PDB_ID"].map(pdb_to_smiles)
    result["SMILES_hits"] = result["SMILES_hits"].apply(lambda x: x if isinstance(x, list) else [])

    return result


all_pairs_df = get_all_pairs()

# Quick summary
print(all_pairs_df.head(10))
print(f"\nTotal proteins: {len(all_pairs_df)}")
print(f"\nUnique proteins: {len(unique_pdb_ids)}")
# dataframes line up ok we're good. great use of 2 hours of my time.

  PDB_ID                                        SMILES_hits
0   7K6Z  [NS(=O)(=O)c1ccc(cc1)-c1ccccc1, Cc1ccccc1-c1cc...
1   6HEU  [Cc1ccc(cc1)-n1nc(cc1NC(=O)Nc1ccc(OCCN2CCOCC2)...
2   7GD6  [O=c1n([se]c2ccccc12)-c1ccccc1, CN(C)CCCSc1ccc...
3   4MJM  [C[C@@H](Oc1cc[n+]([O-])c2ccccc12)c1cn(nn1)-c1...
4   5OL3  [C[C@@]12[C@@H]([C@@H](C[C@@H](O1)n3c4ccccc4c5...
5   3OLS  [Oc1ccc(cc1)-c1sc2cc(O)ccc2c1C(=O)c1ccc(OCCN2C...
6   6BAD  [CCCc1c(O)c(O)c(C(O)=O)c2cc(Cc3ccccc3)c(C)cc12...
7   4CYN  [Cc1nn(C)c(C)c1NS(=O)(=O)c1c(Cl)cc(cc1Cl)-c1cc...
8   7ONQ  [NS(=O)(=O)c1ccc(cc1)-c1ccccc1, Cc1ccccc1-c1cc...
9   2Y82  [Clc1ccc(s1)C(=O)NC[C@H]1CN(C(=O)O1)c1ccc(cc1)...

Total proteins: 20420

Unique proteins: 20420


In [ ]:
all_pairs_df = pd.DataFrame(all_pairs_df)
all_pairs_df.to_csv("notebook_database/all_pairs_df.csv")